In [ ]:
# | default_exp sqlite_db

# DB setup 
> A Simple API for configure the DB connection with Sqlite

In [ ]:
# | export
from sqlmodel import create_engine, Session, SQLModel
import os
from dotenv import load_dotenv

load_dotenv()


In [ ]:
#| export

from contextlib import contextmanager


_engines = {}


@contextmanager
def get_session(db_url: str | None = None  # Full SQLite URL, or uses `SEEOOTTER_DB_URL` env var
                ) -> Session:
    "Create a SQLite session."
    if db_url is None: db_url = os.getenv("SEEOOTTER_DB_URL")
    if db_url not in _engines:
        engine = create_engine(db_url, connect_args={"timeout": 30.0})
        with engine.connect() as conn:
            conn.exec_driver_sql("PRAGMA journal_mode=WAL;")
        SQLModel.metadata.create_all(engine)
        _engines[db_url] = engine
    else:
        engine = _engines[db_url]
    session = Session(engine)
    try:
        yield session
    finally:
        session.close()


In [ ]:
# | hide

with get_session() as session:
    print(session.bind.url)
